# **Import des modules nécessaires**

In [7]:
import pandas as pd #pour la manipulation de données
from pathlib import Path #pour la gestion des chemins de fichiers

from bertopic import BERTopic #pour la modélisation de sujets

from umap import UMAP #pour la réduction de dimensionnalité

from sklearn.cluster import KMeans #pour le clustering
from hdbscan import HDBSCAN #pour le clustering de BERTopic

from sklearn.feature_extraction.text import CountVectorizer #pour la vectorisation de texte
from bertopic.vectorizers import ClassTfidfTransformer #pour la vectorisation de texte spécifique à BERTopic

from sentence_transformers import SentenceTransformer #pour les embeddings de phrases

# **Chargement du corpus de Zola**


In [8]:
df=pd.read_csv(Path("..") /"data" /"2_processed"/ "02_corpus_zola.csv", encoding="utf-8",)
df.head()


,roman,annee,ordre_romans,paquet_id,texte,nb_mots
0,1865 La confession de Claude.,1865,1,1,"Voici l’hiver: l’air, au matin, devient plus f...",985
1,1865 La confession de Claude.,1865,1,2,"Ai-je eu trop de confiance en ma force, ma pla...",1067
2,1865 La confession de Claude.,1865,1,3,Hélas! il me faut cependant une ombre de réali...,894
3,1865 La confession de Claude.,1865,1,4,Elle dormait. J’ai entassé sur ses pieds tous ...,1158
4,1865 La confession de Claude.,1865,1,5,"Ainsi, jamais mon cœur ne pourra battre sans q...",1027


In [9]:
df.shape

(3556, 6)

# **Traitement du Corpus de Zola**

## **1) Choix du modèle d'embedding**

Ici je vais choisir un modèle d'embedding pré-entraîné pour transformer les textes en vecteurs numériques. Je vais utiliser un modèle de la bibliothèque Sentence Transformers, qui est compatible avec BERTopic.

In [11]:
embedding_model = SentenceTransformer(
    "sentence-transformers/distiluse-base-multilingual-cased-v2"
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

## **2) Pipeline de Traitement**

In [12]:
docs = df["texte"].tolist()

timestamps = df["ordre_romans"].tolist()

### 1) UMAP

In [13]:
umap_model = UMAP(
    n_neighbors=15,
    n_components=10,
    min_dist=0.1,
    metric="cosine",
    random_state=42
)

### 2) HDBSCAN

In [14]:
hdbscan_model = HDBSCAN(
    min_cluster_size=15,
    min_samples=1,
    metric="euclidean",
    cluster_selection_method="leaf",
    prediction_data=True
)

### 3) **Kmeans**

In [15]:
cluster_model = KMeans(
    n_clusters=15,
    random_state=42,
    n_init="auto"

)

### 3) creation des stop words + vectorisation TF-IDF

In [16]:
stop_word = [
    # Rougon / Macquart / noms familiaux centraux
    "rougon", "macquart", "saccard", "mouret", "lantier", "coupeau", "maheu", "fouan", "quenu", "chanteau", "josserand", "deberle",
    "raquin", "lorilleux", "buteau", "roubaud", "baudu", "gervaise",

    # Prénoms / personnages très fréquents
    "pierre", "jean", "jacques", "marie", "madeleine", "pauline", "hélène", "jeanne", "claude", "marius", "lazare", "guillaume",
    "laurent", "philippe", "thérèse", "albine", "maurice", "marthe", "maxime", "daniel", "camille", "serge", "octave", "florent",
    "félicité", "louise", "silvère", "caroline", "lisa", "laurence", "miette", "simon", "josine", "clotilde", "véronique", "juliette",
    "rosalie", "mathieu", "charles", "françoise", "lise", "séverine", "henriette", "georges", "angélique", "adèle", "pascal", "lucie", 
    "martine", "virginie", "suzanne", "valentine", "olympe", "lucien", "armande", "constance", "berthe", "sidonie", "benedetta",

    # Noms propres / personnages complémentaires
    "luc", "geneviève", "catherine", "chaval", "adélaïde", "antoine", "auguste", "marianne", "clorinde", "normande", "saget", "mlle saget",
    "delhomme", "ragu", "trouche", "sandoz", "mahoudeau", "bonneville", "rastoil", "rambaud", "fenil", "surin", "malignon", "faloise",
    "lambesc", "mignot", "baugé", "morange", "bouchard", "teuse", "pâquerette", "paquerette", "duparque", "mme duparque", "guersaint",
    "aurélie", "aurelie", "mme aurélie", "mme aurelie", "lecœur", "lecoeur", "mme lecœur", "mme lecoeur", "granoux", "bourron",
    "lerat", "mme lerat", "duveyrier", "bachelard", "gueulin", "delaherche", "beauclair", "lison", "orlando", "delbos",
    "boisgelin", "fernande", "sébastien", "sebastien", "mazaud", "dubuche", "moreux", "vigneron", "bourrette", "satan",
    "archangias", "férat", "ferat", "barbue", "trouille", "laveuve",

    # Personnages secondaires / noms très discriminants
    "fauchery", "muffat", "bordenave", "zoé", "mignon", "faujas", "mathéus", "trublot", "fagerolles", "cazalis", "poizat", "jordan",
    "vandeuvres", "condamin", "paloque", "charbonnel", "gilquin", "marjolin", "campardon", "toussaint", "grivet", "denizet", "kahn",
    "hutin", "bonnaire", "laboque", "marsy", "nanet", "favier", "deloche", "goujet", "cadine", "chouteau", "rochas", "lepailleur",
    "séguin", "gagnière", "hourdequin", "macqueron", "théodose", "savin", "gérard", "larsonneau", "morfain", "lorin", "salvat",
    "boche", "gavard", "martelly",

    # Groupes nominaux religieux / institutionnels
    "sœur hyacinthe", "soeur hyacinthe", "mère chantemesse", "mere chantemesse",

     # Formes d’adresse / famille
    "monsieur", "madame", "mademoiselle", "mme", "mlle",
    "maman", "papa", "père", "pere", "mère", "mere",
    "frère", "frere", "frères", "freres", "sœur", "soeur",
    "sœurette", "soeurette", "tante", "oncle", "cousin",
    "cousine", "époux", "epoux", "épouse", "epouse",
    "nièce", "niece",

    # Mots très fréquents et peu informatifs
    "le", "la", "les", "un", "une", "des", "du", "de", "d", "au", "aux",
    "ce", "cet", "cette", "ces", "son", "sa", "ses", "leur", "leurs",
    "je", "tu", "il", "elle", "nous", "vous", "ils", "elles",
    "me", "te", "se", "moi", "toi", "lui", "eux",
    "que", "qui", "quoi", "dont", "où", "et", "ou", "mais", "donc",
    "or", "ni", "car", "ne", "pas", "plus", "moins", "très",
    "dans", "sur", "sous", "avec", "sans", "pour", "par", "comme",
    "chez", "vers", "entre", "contre", "depuis", "pendant",
    "tout", "tous", "toute", "toutes", "même", "autre", "autres",
    "être", "avoir", "faire", "dire", "aller", "voir", "savoir",
    "pouvoir", "vouloir", "falloir", "devoir", "venir", "prendre",
    "mettre", "donner", "trouver", "passer", "rester", "sembler",
    "encore", "toujours", "jamais", "bien", "mal", "peu", "assez",
    "alors", "puis", "enfin", "aussi", "ainsi", "déjà", "là",
    "ici", "cela", "ça", "ceci", "celui", "celle", "ceux", "celles"
]

In [23]:
vectorizer_model = CountVectorizer(
    stop_words=stop_word,
    min_df=3,
    max_df=0.50
)

ctfidf_model = ClassTfidfTransformer(
    reduce_frequent_words=True
)

### 4) Création du modèle BERTopic

In [24]:
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=cluster_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    language="multilingual",
    calculate_probabilities=False,
    verbose=True
)

topics, probs = topic_model.fit_transform(docs)

df["topic"] = topics

2026-06-02 18:01:29,381 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/112 [00:00<?, ?it/s]

2026-06-02 18:02:22,686 - BERTopic - Embedding - Completed ✓
2026-06-02 18:02:22,687 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-02 18:02:40,915 - BERTopic - Dimensionality - Completed ✓
2026-06-02 18:02:40,918 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-02 18:02:40,951 - BERTopic - Cluster - Completed ✓
2026-06-02 18:02:40,957 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-02 18:02:42,907 - BERTopic - Representation - Completed ✓


## **3) Topics Présent**

In [25]:
topic_info = topic_model.get_topic_info()
topic_info

,Topic,Count,Name,Representation,Representative_Docs
0,0,384,0_loubet_autels_nefs_escouade,"[loubet, autels, nefs, escouade, trouées, lapo...",[Le voilà! Le voilà! Et des poussées se produi...
1,1,329,1_bodin_tellier_lilitte_zéphyrin,"[bodin, tellier, lilitte, zéphyrin, verdier, f...","[Non, maman dit qu’il n’est pas honnête. Ah! t..."
2,2,304,2_zéphyrin_cravache_luigi_plouguern,"[zéphyrin, cravache, luigi, plouguern, pozzo, ...","[Renée était l’homme, la volonté passionnée et..."
3,3,293,3_tellier_tiburce_rionne_rosine,"[tellier, tiburce, rionne, rosine, rieu, souff...",[C’est que Madeleine était un de ces tempérame...
4,4,262,4_tiburce_santerre_rieu_hordel,"[tiburce, santerre, rieu, hordel, tra, lalie, ...","[Et j’allais ainsi, comme dans un rêve, interr..."
5,5,255,5_donadéi_jeanbernat_obligations_fornaro,"[donadéi, jeanbernat, obligations, fornaro, le...",[Je laisserai le meilleur de mon être dans cet...
6,6,243,6_illy_lapoulle_loubet_ducrot,"[illy, lapoulle, loubet, ducrot, pache, douay,...","[Déchaussez-vous, marchez le pied nu, la boue ..."
7,7,231,7_santerre_enclave_polydor_ferrand,"[santerre, enclave, polydor, ferrand, adrien, ...","[Restée seule, Mme Alexandre le continuait dan..."
8,8,222,8_altesse_bijard_goliath_lalie,"[altesse, bijard, goliath, lalie, herscheuse, ...",[Jusqu’aux enfants au maillot qui vont dans le...
9,9,217,9_verlaque_muche_davoine_poissonnières,"[verlaque, muche, davoine, poissonnières, mann...","[Presque guéri de sa blessure au poignet, il s..."


In [22]:
for topic_id in topic_info["Topic"].head(15):
    if topic_id != -1:
        print("\nTOPIC", topic_id)
        print(topic_model.get_topic(topic_id)[:15])


TOPIC 0
[('trône', np.float64(0.16247046029781428)), ('verdures', np.float64(0.15959845253283245)), ('hangar', np.float64(0.15002457274068418)), ('vierges', np.float64(0.14747887965236722)), ('briques', np.float64(0.14483243050384442)), ('denise', np.float64(0.14117383325752925)), ('loubet', np.float64(0.13986064523033023)), ('seine', np.float64(0.13869155860762422)), ('dario', np.float64(0.1371670863777965)), ('bernadette', np.float64(0.13569941455602544))]

TOPIC 1
[('denise', np.float64(0.32347822414143507)), ('clara', np.float64(0.19973811642777692)), ('desforges', np.float64(0.19965791754307358)), ('robineau', np.float64(0.18563972957894695)), ('hortense', np.float64(0.1830359640674534)), ('bodin', np.float64(0.179209511857678)), ('jouve', np.float64(0.17894953136612715)), ('valérie', np.float64(0.17528107484903252)), ('bonamy', np.float64(0.17246798169376099)), ('tellier', np.float64(0.16976511163083802))]

TOPIC 2
[('delestang', np.float64(0.16705528419087587)), ('zéphyrin', np